# CampusEvent AI – אבן דרך 1
**מגישים:** ראיד אבו דבוס, סובחי סלאמה | קורס סוכני AI, תשפ"ז סמסטר א'

המחברת רק מריצה את הקוד שבמאגר (`src/`, `tests/`). ברירת המחדל: **מודל מדומה (MockModel) ושירות יומן מדומה (CampusDB)** – בלי מפתח ובלי עלות.

In [ ]:
# 1) שכפול המאגר
!git clone https://github.com/subhisalami14-cmd/campusevent-ai.git
%cd campusevent-ai

In [ ]:
# 2) התקנת התלויות (גרסאות מדויקות)
!pip install -q -r requirements.txt

In [ ]:
# 3) הרצת כל הבדיקות בפקודה אחת (מודל וכלים מדומים, בלי רשת)
!python -m pytest

In [ ]:
# 4) חוזי הכלים
import json
from src.tools import list_contracts
for c in list_contracts():
    print(f"{c['name']:<22} kind={c['kind']:<8} approval={c['requires_approval']!s:<5} retry={c['retry']}")
print(json.dumps(list_contracts()[0]['input_schema'], ensure_ascii=False, indent=1))

In [ ]:
# 5) הדגמה: הרצה תקינה, כשל זמני, והשוואת הקשר מלא מול מצומצם (שומר עקבות ב-docs/traces)
!python -m src.demo

In [ ]:
# 6) שיחה ידנית עם הסוכן (מודל מדומה)
import datetime as dt
from src.agent import Agent, MockModel
agent = Agent(model=MockModel(today=dt.date(2026, 12, 1)))
for text in ['תאם חדר ל־40 סטודנטים ב־15/12 בין 10:00–12:00 עם מקרן ו־2 מיקרופונים, תקציב עד 800 ש"ח', 'כן']:
    r = agent.run_turn('colab-thread', 'u_colab', text)
    print(f'user: {text}\nagent [{r.status}]: {r.reply}\n')

## מודל אמיתי (אופציונלי)
רץ רק אם הוגדר `GEMINI_API_KEY` ב-**Colab Secrets** (סמל המפתח בסרגל הצד). המפתח לא נשמר במאגר.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    pass

if os.environ.get('GEMINI_API_KEY'):
    !pip install -q google-genai
    from src.agent import Agent, GeminiModel
    real = Agent(model=GeminiModel())
    r = real.run_turn('real-thread', 'u_colab', 'תאם חדר ל־30 איש ב־18/12 בין 12:00–14:00 עם מקרן, תקציב עד 600 ש"ח.')
    print(r.status, r.reply)
else:
    print('אין GEMINI_API_KEY – מדלגים על המודל האמיתי (ההדגמה למעלה רצה עם המודל המדומה).')